In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
ModelName = os.getenv("ModelName")

In [ ]:
from langchain_groq import ChatGroq

LLM = ChatGroq(
    model = ModelName,
    api_key = GroqApiKey
)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

Client = MultiServerMCPClient(
    {
        "search_server": {
            "command": "python",
            "args": ["mcp_servers/SearchServer.py"],
            "transport": "stdio"
        },

        "file_server": {
            "command": "python",
            "args": ["mcp_servers/FileServer.py"],
            "transport": "stdio"
        }
    }
)

In [ ]:
Tools = await Client.get_tools()

print("Loaded Tools:")

for Tool in Tools:
    print(Tool.name)

In [ ]:
from langgraph.prebuilt import create_react_agent

Agent = create_react_agent( model=LLM, tools=Tools )

In [ ]:
os.makedirs("logs", exist_ok=True)

TraceFile = "logs/ToolTrace.txt"

def saveTrace(Content):
    with open( TraceFile, "a", encoding="utf-8" ) as File:
        File.write(Content)
        File.write("\n\n")

In [ ]:
ResearchQuestion = input( "Enter Research Question: " )

In [ ]:
Response = Agent.stream( {"messages": [( "user", ResearchQuestion) ]}, stream_mode="values" )

FinalAnswer = ""

for Step in Response:

    Message = Step["messages"][-1]

    print("\n")
    print(type(Message).__name__)
    print(Message)

    saveTrace(str(Message))

    if hasattr(Message, "content"):
        FinalAnswer = Message.content

In [ ]:
print("\nAnswer")
print(FinalAnswer)